# Prerrequisitos

In [10]:
using LinearAlgebra
using SparseArrays

# Métodos de subspacios de Krylov

Defina el subspacio de Krylov $\mathcal{K}_m$ por
$$
\mathcal{K}_m(A,r)=\mbox{span}\{ r,Ar,A^2r,\dots,A^{m-1}r\} = \{ w=q(A)r \quad : \quad q \in Pol_{m-1}\} 
$$
donde $Pol_{m-1}$ es el conjunto de polinomios degrado menor o igual que $m-1$ en una variable. 

Un método de Krylov es un método de proyección usando el subspacio $\mathcal{K}=\mathcal{K}(A, b-Ax^{(0)})$. Diferemtes métodos de subespcios de Krylov resultan al seleccionar el subspacio de prueba $\mathcal{L}$ y tambien de los diferentes precondicionadores que puedan ser usados. 

Las elecciones mas usadas para $\mathcal{L}$ son, 

1. El subspacios $\mathcal{L}=\mathcal{K}=\mathcal{K}(A, b-Ax^{(0)})$ y su variación de residuo minimo  $\mathcal{L}=A\mathcal{K}=A\mathcal{K}(A, b-Ax^{(0)})$.
2. El subspacio $\mathcal{L}=\mathcal{K}(A^T, b-Ax^{(0)})$.
3. Versiones por bloques de estos subespacios.


Note que estamos intentando de evaluar la aproximación 
$$
A^{-1}\approx x_m =x{(0)}+q_{m-1}(A) \Big( b-Ax^{(0)}\Big)
$$
para algún polinomio de grado a lo sumo $m-1$. Los diferentes espacios $\mathcal{L}$ generan diferentes conjuntos de restricciones para calcular la paroximación.

## Subspacios de Krylov

Tenemos 
$$
\mathcal{K}_m(A,v)=\mbox{span}\{ r,Av,A^2v,\dots,A^{m-1}v\} = \{ w=q(A)v \quad : \quad q \in Pol_{m-1}\} 
$$
El polinomio minimal de un vector $v$ es el polinomio mónico $p$ de menor grado tal que $p(A)v=0$. El grado de este poliniomo es conocido como grado de $v$ con respecto a la matriz $A$ y es denotado por $grad(v)$.

Tenemos  entonces $A\mathcal{K}_{grad(v)}(A,v)\subset \mathcal{K}_{grad(v)}(A,v)$ y para $m\geq grad(v)$ tenemos que 
$$
\mathcal{K}_m(A,v)=\mathcal{K}_{grad(v)}(A,v).
$$
También tenemos que 
$$
dim(\mathcal{K}_m(A,v) = m \mbox{ si y solamente si }  grad(v)\geq m.
$$
En resumen, $dim(\mathcal{K}_m(A,v) =\min\{ m,grad(v)\}.$


# Método de Arnoldi

In [11]:
function pcgmres(r)
    return r
end

pcgmres (generic function with 1 method)

In [12]:
function gmres(A,x,b,restrt,max_it,tol)
#  Translate to julia from
#-- Iterative template routine --
#     Univ. of Tennessee and Oak Ridge National Laboratory
#     October 1, 1993
#     Details of this algorithm are described in "Templates for the
#     Solution of Linear Systems: Building Blocks for Iterative
#     Methods", Barrett, Berry, Chan, Demmel, Donato, Dongarra,
#     Eijkhout, Pozo, Romine, and van der Vorst, SIAM Publications,
#     1993. (ftp netlib2.cs.utk.edu; cd linalg; get templates.ps).
#
# [x, error, iter, flag] = gmres( A, x, b, M, restrt, max_it, tol )
#
# gmres.m solves the linear system Ax=b
# using the Generalized Minimal residual ( GMRESm ) method with restarts .
#
# input   A        REAL nonsymmetric positive definite matrix
#         x        REAL initial guess vector
#         b        REAL right hand side vector
#         M        REAL preconditioner matrix
#         restrt   INTEGER number of iterations between restarts
#         max_it   INTEGER maximum number of iterations
#         tol      REAL error tolerance
#
# output  x        REAL solution vector
#         error    REAL error norm
#         iter     INTEGER number of iterations performed
#         flag     INTEGER: 0 = solution found to tolerance
#                           1 = no convergence given max_it

    iter = 0  # initialization
    flag = 0
    bnrm2 = norm( b );
    if  ( bnrm2 == 0.0 )
            bnrm2 = 1.0
    end
    
    
    #  r = M \ ( b-A*x )
    r = pcgmres( b-A*x )
    println(" Left preconditioned GMRES residual(0) = %g\n",norm( r ))
    error = norm( r ) / bnrm2
    if ( error < tol ) 
        return
    end

    n = size(A)[1]                                #  initialize workspace
    m = restrt
    V= zeros(n,m+1)
    H= zeros(m+1,m)
    cs = zeros(m,1)
    sn= zeros(m,1)
    e1    = zeros(n,1)
    e1[1] = 1.0
    for iter = 1:max_it                              # begin iteration
        println(" iter= ",iter)
        #%     r = M \ ( b-A*x )
        r = pcgmres( b-A*x )
        V[:,1] = r / norm( r )
        s = norm( r )*e1
        
        for i = 1:m                                   # construct orthonormal
            #% w = M \ (A*V(:,i));                    # basis using Gram-Schmidt
            w = pcgmres(A*V[:,i])                          # basis using Gram-Schmidt
            for k = 1:i
                H[k,i]= w'V[:,k]
                w = w - H[k,i]*V[:,k]
            end
            H[i+1,i] = norm( w )
            V[:,i+1] = w / H[i+1,i]
            for k = 1:i-1                              # apply Givens rotation
                temp     =  cs[k]*H[k,i] + sn[k]*H[k+1,i]
                H[k+1,i] = -sn[k]*H[k,i] + cs[k]*H[k+1,i]
                H[k,i]   = temp
            end
     
            γ=1.0/norm([H[i:i+1,i]])
            cs[i]=H[i,i]*γ
            sn[i] = H[i+1,i]*γ # form i-th rotation matrix
            
            temp   = cs[i]*s[i]                       # approximate residual norm
            s[i+1] = -sn[i]*s[i]
            s[i]   = temp
            H[i,i] = cs[i]*H[i,i] + sn[i]*H[i+1,i]
            H[i+1,i] = 0.0
 
            error  = abs(s[i+1]) / bnrm2
            println("Left preconditioned GMRES residual",i," = ",error)
            if ( error <= tol )                       # update approximation
                y = H[1:i,1:i] \ s[1:i]               # and exit
                x = x + V[:,1:i]*y
                break
            end
        end
        println("Left preconditioned GMRES residual",iter," = ",error)

        if ( error <= tol )
                break
        end
        y = H[1:m,1:m] \ s[1:m]
        x = x + V[:,1:m]*y                       # update approximation

        #     r = M \ ( b-A*x )                   # compute residual
        r = pcgmres( b-A*x )                            # compute residual
        s[m+1] = norm(r)
        error = s[m+1] / bnrm2                      # check convergence
        println(" Left preconditioned GMRES residual",iter," = ",error)
        if ( error <= tol ) 
                break
        end    
    end    
    
    if ( error > tol ) 
            flag = 1
    end                 # converged


    
    return [x, error, iter, flag]   
end

gmres (generic function with 1 method)

In [13]:
n=100
A=rand(n,n)
b=fill(1,(n,1))
x=b*0;

In [14]:
x, error, iter, flag  =gmres(A,x,b,90,100,0.001)

 Left preconditioned GMRES residual(0) = %g
10.0
 iter= 1
Left preconditioned GMRES residual1 = 0.0565069487329334
Left preconditioned GMRES residual2 = 0.05647840000279443
Left preconditioned GMRES residual3 = 0.05609541197630365
Left preconditioned GMRES residual4 = 0.05608522697295609
Left preconditioned GMRES residual5 = 0.05524458682971064
Left preconditioned GMRES residual6 = 0.05522702303843866
Left preconditioned GMRES residual7 = 0.05521103383292425
Left preconditioned GMRES residual8 = 0.05481087267935867
Left preconditioned GMRES residual9 = 0.05475476683670335
Left preconditioned GMRES residual10 = 0.05462347154995969
Left preconditioned GMRES residual11 = 0.05452998007609566
Left preconditioned GMRES residual12 = 0.054528640269254844
Left preconditioned GMRES residual13 = 0.05452159597292529
Left preconditioned GMRES residual14 = 0.05447359251744446
Left preconditioned GMRES residual15 = 0.05438921914205179
Left preconditioned GMRES residual16 = 0.05426007964254281
Left pr

Left preconditioned GMRES residual27 = 0.018019095531671477
Left preconditioned GMRES residual28 = 0.018019095520385924
Left preconditioned GMRES residual29 = 0.018019095293575822
Left preconditioned GMRES residual30 = 0.018019087054254347
Left preconditioned GMRES residual31 = 0.018019070486420045
Left preconditioned GMRES residual32 = 0.01801907035069059
Left preconditioned GMRES residual33 = 0.018019048735464553
Left preconditioned GMRES residual34 = 0.018019046281378077
Left preconditioned GMRES residual35 = 0.01801904262999758
Left preconditioned GMRES residual36 = 0.018019030295119635
Left preconditioned GMRES residual37 = 0.018019027914044593
Left preconditioned GMRES residual38 = 0.01801902470583192
Left preconditioned GMRES residual39 = 0.018019018780224254
Left preconditioned GMRES residual40 = 0.018019018036949525
Left preconditioned GMRES residual41 = 0.018019004251253936
Left preconditioned GMRES residual42 = 0.018018957365409736
Left preconditioned GMRES residual43 = 0.01

Left preconditioned GMRES residual15 = 0.017276229055790033
Left preconditioned GMRES residual16 = 0.01727622358648109
Left preconditioned GMRES residual17 = 0.017276216314266173
Left preconditioned GMRES residual18 = 0.017276198542122624
Left preconditioned GMRES residual19 = 0.017276192074469603
Left preconditioned GMRES residual20 = 0.017276190446340173
Left preconditioned GMRES residual21 = 0.0172761862603874
Left preconditioned GMRES residual22 = 0.01727618408146858
Left preconditioned GMRES residual23 = 0.0172761826026662
Left preconditioned GMRES residual24 = 0.017276095023779872
Left preconditioned GMRES residual25 = 0.017276079293636813
Left preconditioned GMRES residual26 = 0.017276067007684654
Left preconditioned GMRES residual27 = 0.017276066644963815
Left preconditioned GMRES residual28 = 0.01727606203123933
Left preconditioned GMRES residual29 = 0.017276061900373077
Left preconditioned GMRES residual30 = 0.017276057077508976
Left preconditioned GMRES residual31 = 0.017276

Left preconditioned GMRES residual4 = 0.017255491273399585
Left preconditioned GMRES residual5 = 0.017255491254333434
Left preconditioned GMRES residual6 = 0.017255491157518798
Left preconditioned GMRES residual7 = 0.017255488995373054
Left preconditioned GMRES residual8 = 0.017255488952489694
Left preconditioned GMRES residual9 = 0.017255488896204173
Left preconditioned GMRES residual10 = 0.01725548884620693
Left preconditioned GMRES residual11 = 0.017255488649631636
Left preconditioned GMRES residual12 = 0.017255488547521864
Left preconditioned GMRES residual13 = 0.017255488522230165
Left preconditioned GMRES residual14 = 0.01725548789457723
Left preconditioned GMRES residual15 = 0.017255487270577567
Left preconditioned GMRES residual16 = 0.01725548677318773
Left preconditioned GMRES residual17 = 0.017255486404051147
Left preconditioned GMRES residual18 = 0.017255485140834782
Left preconditioned GMRES residual19 = 0.017255484627418542
Left preconditioned GMRES residual20 = 0.01725548

Left preconditioned GMRES residual49 = 0.01724384997803083
Left preconditioned GMRES residual50 = 0.01724384984727222
Left preconditioned GMRES residual51 = 0.01724384964250226
Left preconditioned GMRES residual52 = 0.017243844377938723
Left preconditioned GMRES residual53 = 0.01724384008714661
Left preconditioned GMRES residual54 = 0.01724383653682351
Left preconditioned GMRES residual55 = 0.01724383649766673
Left preconditioned GMRES residual56 = 0.017243836469757863
Left preconditioned GMRES residual57 = 0.01724383539757449
Left preconditioned GMRES residual58 = 0.01724383523163206
Left preconditioned GMRES residual59 = 0.01724383477229175
Left preconditioned GMRES residual60 = 0.017243834772276548
Left preconditioned GMRES residual61 = 0.017243825698741184
Left preconditioned GMRES residual62 = 0.017243816990606876
Left preconditioned GMRES residual63 = 0.017243795429161685
Left preconditioned GMRES residual64 = 0.01724377845866442
Left preconditioned GMRES residual65 = 0.017243778

...

Left preconditioned GMRES residual64 = 0.01599481420007318
Left preconditioned GMRES residual65 = 0.015994814200073176
Left preconditioned GMRES residual66 = 0.015994814200073156
Left preconditioned GMRES residual67 = 0.01599481420007301
Left preconditioned GMRES residual68 = 0.015994814200072878
Left preconditioned GMRES residual69 = 0.015994814200072857
Left preconditioned GMRES residual70 = 0.01599481420007285
Left preconditioned GMRES residual71 = 0.015994814200072777
Left preconditioned GMRES residual72 = 0.015994814200072767
Left preconditioned GMRES residual73 = 0.01599481420007275
Left preconditioned GMRES residual74 = 0.015994814200072732
Left preconditioned GMRES residual75 = 0.015994814200072625
Left preconditioned GMRES residual76 = 0.01599481420007259
Left preconditioned GMRES residual77 = 0.01599481420007259
Left preconditioned GMRES residual78 = 0.015994814200072573
Left preconditioned GMRES residual79 = 0.015994814200072503
Left preconditioned GMRES residual80 = 0.01599

Left preconditioned GMRES residual53 = 0.015994814200049782
Left preconditioned GMRES residual54 = 0.01599481420004977
Left preconditioned GMRES residual55 = 0.01599481420004977
Left preconditioned GMRES residual56 = 0.01599481420004977
Left preconditioned GMRES residual57 = 0.01599481420004977
Left preconditioned GMRES residual58 = 0.015994814200049765
Left preconditioned GMRES residual59 = 0.015994814200049765
Left preconditioned GMRES residual60 = 0.015994814200049765
Left preconditioned GMRES residual61 = 0.015994814200049726
Left preconditioned GMRES residual62 = 0.01599481420004967
Left preconditioned GMRES residual63 = 0.015994814200049615
Left preconditioned GMRES residual64 = 0.01599481420004955
Left preconditioned GMRES residual65 = 0.015994814200049546
Left preconditioned GMRES residual66 = 0.015994814200049532
Left preconditioned GMRES residual67 = 0.01599481420004945
Left preconditioned GMRES residual68 = 0.01599481420004937
Left preconditioned GMRES residual69 = 0.0159948

Excessive output truncated after 524305 bytes.

Left preconditioned GMRES residual3 = 0.015994814200042135
Left preconditioned GMRES residual4 = 0.015994814200042125
Left preconditioned GMRES residual5 = 0.015994814200042125
Left preconditioned GMRES residual6 = 0.015994814200042125
Left preconditioned GMRES residual7 = 0.01599481420004212
Left preconditioned GMRES residual8 = 0.01599481420004212
Left preconditioned GMRES residual9 = 0.015994814200042118
Left preconditioned GMRES residual10 = 0.015994814200042118
Left preconditioned GMRES residual11 = 0.015994814200042118
Left preconditioned GMRES residual12 = 0.015994814200042118
Left preconditioned GMRES residual13 = 0.015994814200042114
Left preconditioned GMRES residual14 = 0.015994814200042114
Left preconditioned GMRES residual15 = 0.01599481420004211
Left preconditioned GMRES residual16 = 0.01599481420004211
Left preconditioned GMRES residual17 = 0.01599481420004211
Left preconditioned GMRES residual18 = 0.01599481420004211
Left preconditioned GMRES residual19 = 0.015994814200

4-element Vector{Any}:
  [-0.07031125851508921; 0.12215906709561365; … ; -0.05861598799252555; -0.11952225494253288;;]
 0.01599481420002243
 0
 1

In [15]:
b-A*x

100×1 Matrix{Float64}:
 -0.009890826355506865
 -0.019659410466125404
 -0.008033608282792315
 -0.006195532675086657
  0.008529800387263586
 -0.010031384733429904
  0.013863615946806296
  0.01991898701604433
  0.006887964556716586
 -0.009130101803418222
 -0.02539949178355183
 -0.026642046111994944
  0.005850663508607745
  ⋮
  0.0036666526209925276
 -0.003947985763876449
  0.0196317380729637
 -0.006532109503023964
  0.022032944326254578
  0.004411326166118723
 -0.02582174387317049
 -0.014933597946068744
  0.019071602000867105
 -0.011163602094291347
 -0.006218846760814634
 -0.018168865888479546

In [16]:
function pccg(r)
    return r
end

pccg (generic function with 1 method)

In [17]:
function  cg(A, x, b, max_it, tol)
# Translate to julia from
#  -- Iterative template routine --
#     Univ. of Tennessee and Oak Ridge National Laboratory
#     October 1, 1993
#     Details of this algorithm are described in "Templates for the
#     Solution of Linear Systems: Building Blocks for Iterative
#     Methods", Barrett, Berry, Chan, Demmel, Donato, Dongarra,
#     Eijkhout, Pozo, Romine, and van der Vorst, SIAM Publications,
#     1993. (ftp netlib2.cs.utk.edu; cd linalg; get templates.ps).
#
#  [x, error, iter, flag] = cg(A, x, b, M, max_it, tol)
#
# cg.m solves the symmetric positive definite linear system Ax=b 
# using the Conjugate Gradient method with preconditioning.
#
# input   A        REAL symmetric positive definite matrix
#         x        REAL initial guess vector
#         b        REAL right hand side vector
#         M        REAL preconditioner matrix
#         max_it   INTEGER maximum number of iterations
#         tol      REAL error tolerance
#
# output  x        REAL solution vector
#         error    REAL error norm
#         iter     INTEGER number of iterations performed
#         flag     INTEGER: 0 = solution found to tolerance
#                           1 = no convergence given max_it

    flag = 0 # initialization
    iter = 0
    lastiter=max_it
    α=fill(0.0,(1,max_it))
    β=fill(0.0,(1,max_it))
    bnrm2 = norm( b )
    if  ( bnrm2 == 0.0 )
        bnrm2 = 1.0
    end

    r = b - A*x
    error = norm( r ) / bnrm2
    println(" PCG residual ",iter," = ",error)
    if ( error < tol ) 
        return
    end

    for iter = 1:max_it                       # begin iteration

        z  = r
        #    z  = pccg(r,iter)
        rho = (r'*z)
        rho_1=rho
        p=z
        if ( iter > 1 )                       # direction vector
            β[iter] = rho / rho_1
            p = z + β[iter]*p
         else
            p = z
         end

         q = A*p
         α[iter] = rho / (p'*q )
         x = x + α[iter] * p                  # update approximation vector

         r = r - α[iter]*q                    # compute residual
         error = norm( r ) / bnrm2            # check convergence
        println("PCG residual (",iter,") = ",error)
     if ( error <= tol )
            lastiter=iter
            break
    end 



  end
#
#  compute eigenvalues and codition number
#
    d=sparse(fill(0.0, (lastiter,1)))
     d[1]= 1/α[1];
     for i = 2: iter
         d[i]=β[i]/α[i-1]+1/α[i];
     end
     for i = 1: iter-1
         s[i]=-1*sqrt(β[i+1])/α[i];
     end
# 
     T = sparse(fill(0.0, (lastiter,lastiter)))
     T[1,1]= d[1];
     for i=2:iter
         T[i,i] = d[i]
     end
     for i = 1:iter-1
         T[i,i+1]= s[i]; T[i+1,i] = T[i,i+1];
     end
     lambda = eigvals(Matrix(T))
     lambdamax = maximum(lambda)
     lambdamin = minimum(lambda)
     condnumber = lambdamax/lambdamin

    if ( error > tol ) 
        flag = 1
    end         # no convergence
    return [x, error, iter, flag,condnumber]
end

cg (generic function with 1 method)

In [18]:
n=1000
A=rand(n,n)
A=A'A+0.1*I
b=fill(1,n)
x=b*0;

In [ ]:
x, error, iter, flag,cond  =cg(A,x,b,100000,0.000001);
cond

 PCG residual 0 = 1.0
PCG residual (1) = 0.01826764504699618
PCG residual (2) = 0.25421469482402054
PCG residual (3) = 0.1273137478217249
PCG residual (4) = 0.06407196037220045
PCG residual (5) = 0.032885256059054825
PCG residual (6) = 0.018309236156634948
PCG residual (7) = 0.014931788455166363
PCG residual (8) = 0.029976351541236578
PCG residual (9) = 0.01710405633685462
PCG residual (10) = 0.015957537381311483
PCG residual (11) = 0.018724093214446364
PCG residual (12) = 0.014691053756017598
PCG residual (13) = 0.03889917252101391
PCG residual (14) = 0.020935486329194945
PCG residual (15) = 0.014413696618060266
PCG residual (16) = 0.12498489330891549
PCG residual (17) = 0.06288958513557001
PCG residual (18) = 0.032256883454312255
PCG residual (19) = 0.01790892120059293
PCG residual (20) = 0.014391937460697909
PCG residual (21) = 0.031757424873127466
PCG residual (22) = 0.017689538258737623
PCG residual (23) = 0.014462027955335717
PCG residual (24) = 0.028617867563512714
PCG residual 

PCG residual (233) = 0.007889469326381227
PCG residual (234) = 0.019111892551797648
PCG residual (235) = 0.01044564608790139
PCG residual (236) = 0.007751074874098617
PCG residual (237) = 0.04207867071536515
PCG residual (238) = 0.021401824118508845
PCG residual (239) = 0.011471503207665351
PCG residual (240) = 0.007747537957387409
PCG residual (241) = 0.04021981010890022
PCG residual (242) = 0.020489470114138247
PCG residual (243) = 0.011056409149573608
PCG residual (244) = 0.007708727506161898
PCG residual (245) = 0.13271717972509997
PCG residual (246) = 0.066468852535824
PCG residual (247) = 0.03345623628863575
PCG residual (248) = 0.01718226513478437
PCG residual (249) = 0.00959180547768893
PCG residual (250) = 0.007934019181689674
PCG residual (251) = 0.01473150464534444
PCG residual (252) = 0.008614938190502448
PCG residual (253) = 0.00932236026498517
PCG residual (254) = 0.008134096927025294
PCG residual (255) = 0.01184698930595341
PCG residual (256) = 0.007749979544647575
PCG r

PCG residual (545) = 0.009161758802662434
PCG residual (546) = 0.005881203968836285
PCG residual (547) = 0.013780331809072045
PCG residual (548) = 0.00758046325205699
PCG residual (549) = 0.005810241330511976
PCG residual (550) = 0.01950433481634756
PCG residual (551) = 0.010204875701755082
PCG residual (552) = 0.006173630546809878
PCG residual (553) = 0.008429877852274907
PCG residual (554) = 0.005759364405584229
PCG residual (555) = 0.04042340232770025
PCG residual (556) = 0.02041877097946745
PCG residual (557) = 0.010638752064817254
PCG residual (558) = 0.00631887320119209
PCG residual (559) = 0.00757042174117484
PCG residual (560) = 0.005808512686559132
PCG residual (561) = 0.01927241820375661
PCG residual (562) = 0.01009459726190116
PCG residual (563) = 0.006136623650768218
PCG residual (564) = 0.008692583487683661
PCG residual (565) = 0.005788730324073007
PCG residual (566) = 0.022702873621662338
PCG residual (567) = 0.011732729840330738
PCG residual (568) = 0.006726557452805098


PCG residual (840) = 0.007199605744581111
PCG residual (841) = 0.0045211044310227285
PCG residual (842) = 0.008436606219605456
PCG residual (843) = 0.004925533187520867
PCG residual (844) = 0.005274743276875977
PCG residual (845) = 0.004676029475945489
PCG residual (846) = 0.0064271212816456365
PCG residual (847) = 0.004370147096300423
PCG residual (848) = 0.02681449026406735
PCG residual (849) = 0.013587632994147957
PCG residual (850) = 0.007172154169949025
PCG residual (851) = 0.004513273728249151
PCG residual (852) = 0.00859130755228145
PCG residual (853) = 0.00498324606308404
PCG residual (854) = 0.0051251438292989554
PCG residual (855) = 0.004859704920313351
PCG residual (856) = 0.005473927273912547
PCG residual (857) = 0.004517040956205235
PCG residual (858) = 0.008499172541119508
PCG residual (859) = 0.004948423220613585
PCG residual (860) = 0.00520892717562082
PCG residual (861) = 0.004746026618271167
PCG residual (862) = 0.005966580319885842
PCG residual (863) = 0.004363784742

PCG residual (1140) = 0.004179297548165865
PCG residual (1141) = 0.021668095163225747
PCG residual (1142) = 0.011038249365095744
PCG residual (1143) = 0.005955736915217523
PCG residual (1144) = 0.004150166352307632
PCG residual (1145) = 0.06627055227256447
PCG residual (1146) = 0.03319712477548575
PCG residual (1147) = 0.01672306639495601
PCG residual (1148) = 0.008617188867448222
PCG residual (1149) = 0.004878980066105412
PCG residual (1150) = 0.004358264403243962
PCG residual (1151) = 0.005833860380121872
PCG residual (1152) = 0.004045429584908993
PCG residual (1153) = 0.049612421411772335
PCG residual (1154) = 0.02488697787055202
PCG residual (1155) = 0.01260687279782208
PCG residual (1156) = 0.00664570184073058
PCG residual (1157) = 0.004157166582546009
PCG residual (1158) = 0.007480300335545167
PCG residual (1159) = 0.004422816510172417
PCG residual (1160) = 0.005139331028325581
PCG residual (1161) = 0.004080190846550817
PCG residual (1162) = 0.009858696797440544
PCG residual (116

...

PCG residual (11385) = 0.00011068871493874363
PCG residual (11386) = 0.00016124849560575228
PCG residual (11387) = 0.00010547451418012243
PCG residual (11388) = 0.00031278918143659855
PCG residual (11389) = 0.00016582210621489881
PCG residual (11390) = 0.00010642557843257332
PCG residual (11391) = 0.00024883762501034677
PCG residual (11392) = 0.00013694350522456796
PCG residual (11393) = 0.00010519641601539177
PCG residual (11394) = 0.0003445078816613956
PCG residual (11395) = 0.00018067696631408976
PCG residual (11396) = 0.00011045582623020123
PCG residual (11397) = 0.00016348188713911379
PCG residual (11398) = 0.00010591601189384273
PCG residual (11399) = 0.00027696894962217684
PCG residual (11400) = 0.0001494089981497875
PCG residual (11401) = 0.00010401342846189591
PCG residual (11402) = 0.0016409820803568776
PCG residual (11403) = 0.0008221418646339557
PCG residual (11404) = 0.00041439594409149055
PCG residual (11405) = 0.00021404002842087641
PCG residual (11406) = 0.0001224234899

PCG residual (11622) = 0.0005299182327533024
PCG residual (11623) = 0.00026655551630750936
PCG residual (11624) = 0.0001365388853217631
PCG residual (11625) = 7.538553500843138e-5
PCG residual (11626) = 5.887310363758427e-5
PCG residual (11627) = 0.00016335965842556753
PCG residual (11628) = 8.735244503087329e-5
PCG residual (11629) = 5.833788882252685e-5
PCG residual (11630) = 0.00024099208433723624
PCG residual (11631) = 0.0001241330169306079
PCG residual (11632) = 7.01696369659936e-5
PCG residual (11633) = 6.212399826971989e-5
PCG residual (11634) = 8.578152708180471e-5
PCG residual (11635) = 5.8136502810904966e-5
PCG residual (11636) = 0.0003281474861704314
PCG residual (11637) = 0.00016668960166841296
PCG residual (11638) = 8.888020673773228e-5
PCG residual (11639) = 5.8578646290023635e-5
PCG residual (11640) = 0.00019387701698647742
PCG residual (11641) = 0.00010157483249730612
PCG residual (11642) = 6.181606708491007e-5
PCG residual (11643) = 8.830397714656134e-5
PCG residual (1

PCG residual (11866) = 5.041785341979525e-5
PCG residual (11867) = 4.184408640918436e-5
PCG residual (11868) = 7.632649366423522e-5
PCG residual (11869) = 4.491243392896012e-5
PCG residual (11870) = 5.056933478957764e-5
PCG residual (11871) = 4.1750840852573156e-5
PCG residual (11872) = 7.833809609533826e-5
PCG residual (11873) = 4.5652671825672746e-5
PCG residual (11874) = 4.833451399530774e-5
PCG residual (11875) = 4.36273862799548e-5
PCG residual (11876) = 5.647015490539144e-5
PCG residual (11877) = 4.024581420042192e-5
PCG residual (11878) = 0.0012866907521036182
PCG residual (11879) = 0.0006436596872133056
PCG residual (11880) = 0.000322459541850125
PCG residual (11881) = 0.00016249783259355738
PCG residual (11882) = 8.385621024865631e-5
PCG residual (11883) = 4.7777843163233725e-5
PCG residual (11884) = 4.421935477532919e-5
PCG residual (11885) = 5.3111250099663303e-5
PCG residual (11886) = 4.0641813043702526e-5
PCG residual (11887) = 0.00013906691912545755
PCG residual (11888) =

Excessive output truncated after 524289 bytes.

PCG residual (11936) = 4.9776953887215966e-5
PCG residual (11937) = 4.2155513089116765e-5
PCG residual (11938) = 6.959450395623194e-5
PCG residual (11939) = 4.2615164309384645e-5
PCG residual (11940) = 6.38914981785865e-5
PCG residual (11941) = 4.108453940526564e-5
PCG residual (11942) = 9.819212662023957e-5
PCG residual (11943) = 5.3805818365575824e-5
PCG residual (11944) = 4.044368702548741e-5
PCG residual (11945) = 0.00017578512952882877
PCG residual (11946) = 9.028199396264904e-5
PCG residual (11947) = 5.040669007481795e-5
PCG residual (11948) = 4.1729525783456276e-5
PCG residual (11949) = 7.714985816418863e-5
PCG residual (11950) = 4.518450182334829e-5
PCG residual (11951) = 4.936282768207033e-5
PCG residual (11952) = 4.247624382950558e-5
PCG residual (11953) = 6.540217561001577e-5
PCG residual (11954) = 4.1440974205454815e-5
PCG residual (11955) = 8.444930056376794e-5
PCG residual (11956) = 4.800449875984556e-5
PCG residual (11957) = 4.3848947737149635e-5
PCG residual (11958) = 5

In [ ]:
b-A*x